# 1. Deepfake Image Detection Model Training
This notebook demonstrates how to construct, train, and export a Convolutional Neural Network (CNN) classifier to detect face manipulation (deepfakes).

### Objective:
- Process facial images and detect anomalies.
- Train a transfer-learning binary classifier using **MobileNetV2**.
- Export the trained model to `deepfake_model.h5` in the python service directory.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)


## 2. Mock Dataset Generation
To ensure this notebook is executable and self-contained, we generate a synthetic balanced dataset representing facial cropped regions:
- **Real Images (Class 0)**: Simulates smooth, natural facial pixel profiles.
- **Fake Images (Class 1)**: Injects periodic noise and high-frequency patterns mimicking generative artifacts.

In [ ]:
# Generate synthetic dataset of 200 images of size 224x224x3
np.random.seed(42)
num_samples = 100
img_height, img_width = 224, 224

X = np.zeros((num_samples, img_height, img_width, 3), dtype=np.float32)
y = np.zeros((num_samples,), dtype=np.float32)

for i in range(num_samples):
    base = np.random.uniform(0.3, 0.7, (3,))
    img = np.ones((img_height, img_width, 3)) * base
    img[80:120, 60:160, :] = 0.1
    
    if i % 2 == 1:
        noise = np.sin(np.linspace(0, 10 * np.pi, img_height))
        noise_grid, _ = np.meshgrid(noise, noise)
        img[:, :, 0] += noise_grid * 0.15
        y[i] = 1.0
    else:
        img += np.random.normal(0, 0.02, img.shape)
        y[i] = 0.0
        
    X[i] = np.clip(img, 0.0, 1.0)

split_idx = int(num_samples * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Dataset generated: X_train shape {X_train.shape}, y_train shape {y_train.shape}")


## 3. Model Construction
We build the classification network using **MobileNetV2** pre-trained on ImageNet.
We freeze the base convolutional layers to preserve pre-trained features and append:
1. `GlobalAveragePooling2D` layer.
2. `Dropout` layer (rate=0.5) to avoid overfitting.
3. `Dense` output layer with a `sigmoid` activation function for binary classification.

In [ ]:
# Build MobileNetV2 base model with fallback to random weights if offline
try:
    print("Attempting to load MobileNetV2 with ImageNet weights...")
    base_model = MobileNetV2(input_shape=(img_height, img_width, 3), include_top=False, weights='imagenet')
except Exception as e:
    print(f"Internet offline or failed to fetch weights ({e}). Initializing with random weights.")
    base_model = MobileNetV2(input_shape=(img_height, img_width, 3), include_top=False, weights=None)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

print("Model compilation complete.")


## 4. Model Training & Evaluation
We train the compiled model on our synthetic train partition for 3 epochs and evaluate its accuracy and loss curves on validation splits.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=16,
    validation_data=(X_test, y_test)
)

# Plot curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Curves')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy Curves')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Model Export
We export the trained Keras model structure and parameters to the `python/` directory. The Flask microservice will automatically load it.

In [ ]:
output_dir = '../python'
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, 'deepfake_model.h5')
model.save(model_path)
print(f"Model saved successfully to {model_path}")
